In [1]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, make_scorer
import warnings
import re

from sklearn.multioutput import MultiOutputClassifier
from catboost import CatBoostClassifier

In [2]:
warnings.filterwarnings('ignore') 
ohe = OneHotEncoder(sparse_output=False)

In [3]:
df = pd.read_excel("dataset/rigbuilder_main_dataset.xlsx")
df.columns = df.columns.str.strip()
df.drop(columns=['used_amount_saved', 'build_type'], axis=1, inplace=True)
df = df[df['gpu'] != 0].reset_index(drop=True)

In [4]:
df['used_parts'] = df['used_parts'].replace({
    'yes ':'yes', 'New':'no','No':'no','Yes':'yes',
    'Under Warranty Used CPU GPU Storage ':'yes',
    'Under Warranty Used CPU Storage ':'yes', 'Under Warranty Used CPU Storage ':'yes',
    'Under Warranty Used CPU GPU Storage':'yes',
    'Under Warranty Used CPU Storage':'yes'})


df['monitor_required'] = df['monitor_required'].replace({'No':'no','Yes':'yes'})
df['upgrade_path'] = df['upgrade_path'].replace({'Medium':'medium','Low':'Small',
                                                 'good':'medium'})

df['monitor_resolution'] = df['monitor_resolution'].replace({
    'FHD ': 'FHD',
    '4K (UHD)': '4K',
    '1440p (QHD)': 'QHD',
    'QHD Ultrawide': 'UW-QHD',
})

In [5]:
df.dropna(inplace=True)
df1 = df.copy()
df2 = df.copy()
df = pd.concat([df1, df2], axis=0)

In [6]:
cabinet_df = pd.read_excel("dataset/rigbuilder_cabinet.xlsx")
clr_df = pd.read_excel("dataset/rigbuilder_cpu_cooler.xlsx")
cpu_df = pd.read_excel("dataset/rigbuilder_cpu.xlsx")
mobo_df = pd.read_excel("dataset/rigbuilder_mobo.xlsx")
monitor_df = pd.read_excel("dataset/rigbuilder_monitor.xlsx")
psu_df = pd.read_excel("dataset/rigbuilder_psu.xlsx")
ram_df = pd.read_excel("dataset/rigbuilder_ram.xlsx")
storage_df = pd.read_excel("dataset/rigbuilder_storage.xlsx")
gpu_df = pd.read_excel("dataset/rigbuilder_gpu.xlsx")


In [7]:
# Remove extra spaces from all column names

cpu_df.columns = cpu_df.columns.str.strip()
gpu_df.columns = gpu_df.columns.str.strip()
mobo_df.columns = mobo_df.columns.str.strip()
clr_df.columns = clr_df.columns.str.strip()
psu_df.columns = psu_df.columns.str.strip()
storage_df.columns = storage_df.columns.str.strip()
ram_df.columns = ram_df.columns.str.strip()
cabinet_df.columns = cabinet_df.columns.str.strip()
monitor_df.columns = monitor_df.columns.str.strip()


# new
# RAM type format fix
mobo_df['ram_type'] = mobo_df['ram_type'].replace({
    'DDR-3': 'DDR3',
    'DDR-4': 'DDR4',
    'DDR-5': 'DDR5'
})
# new
# Socket format consistent banao (space hatao, uppercase karo)
mobo_df['socket_no'] = mobo_df['socket_no'].str.replace(' ', '', regex=False).str.upper()
cpu_df['socket_no'] = cpu_df['socket_no'].str.replace(' ', '', regex=False).str.upper()
ram_df['mobo_socket_compatibility'] = ram_df['mobo_socket_compatibility'].str.replace(' ', '', regex=False).str.upper()
clr_df['Compatible Socket'] = clr_df['Compatible Socket'].str.replace(' ', '', regex=False).str.upper()
mobo_df['chipset'] = mobo_df['chipset'].str.strip().str.upper()

In [8]:
# x = df[['cpu', 'gpu', 'mobo', 'cpu_cooler', 'psu', 'storage', 'ram', 'cabinet', 'monitor']].iloc[0].tolist()


In [9]:
df.drop(1, inplace=True)
df.reset_index(drop=True, inplace=True)

In [10]:
df['upgrade_path'].replace({'small ': 'small'}, inplace=True)
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')

In [11]:
df["used_parts"] = df["used_parts"].map({
    "yes": 1,
    "no": 0})

df["monitor_required"] = df["monitor_required"].map({
    "yes": 1,
    "no": 0})

df["monitor_size"] = (
    df["monitor_size"]
    .replace(["none", "no"], np.nan)
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
    .fillna(0))

df["monitor_size"] = df["monitor_size"].astype(int)

In [12]:
cols = ['cpu', 'gpu', 'mobo', 'cpu_cooler', 'psu', 'storage', 'ram', 'cabinet', 'monitor']

In [13]:
df = df.dropna(subset=cols)
df.reset_index(drop=True, inplace=True)
print("df:", df.shape)

df: (1350, 17)


In [14]:
# 1. Original cpu_df load karo (agar already filter kar diya hai toh dobara load karo)
cpu_df = pd.read_excel("dataset/rigbuilder_cpu.xlsx")
cpu_df.columns = cpu_df.columns.str.strip()

# 2. Purane socket wale CPU IDs nikaalo
old_sockets = ['LGA1155', 'LGA1150', 'LGA1151']
old_cpu_ids = cpu_df[cpu_df['socket_no'].isin(old_sockets)]['cpu_id'].tolist()
print("Removing CPUs:", old_cpu_ids)

# 3. Ab cpu_df se hatao
cpu_df = cpu_df[~cpu_df['socket_no'].isin(old_sockets)]
cpu_df.reset_index(drop=True, inplace=True)

# 4. Main training dataset (df) se bhi corresponding rows hatao
df = df[~df['cpu'].isin(old_cpu_ids)]
df.reset_index(drop=True, inplace=True)

print("Final training rows:", df.shape)

Removing CPUs: []
Final training rows: (1350, 17)


In [15]:
X = df[['budget', 'used_parts', 'usage_scenario', 'colour_theme',
       'monitor_required', 'monitor_size', 'monitor_resolution',
       'upgrade_path']]

Y = df[['cpu', 'gpu', 'mobo', 'cpu_cooler', 'psu', 'storage',
       'ram', 'cabinet', 'monitor']]

In [16]:
categorical_cols = ["usage_scenario", "colour_theme", "monitor_resolution", "upgrade_path"]
df[categorical_cols] = df[categorical_cols].astype(str)

In [17]:
encoded_data = ohe.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded_data, columns=ohe.get_feature_names_out(categorical_cols))

In [18]:
X = X.drop(columns=categorical_cols)
X = pd.concat([X, encoded_df], axis=1)

In [19]:
target_encoders = {}
for col in Y.columns:
    label_encoder = LabelEncoder()
    Y[col] = label_encoder.fit_transform(Y[col])
    target_encoders[col] = label_encoder


In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

## **Random Forest Model**

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]}

rf = RandomForestClassifier(random_state=42)

search_best_pera = GridSearchCV(estimator=rf, param_grid=param_dist, cv=5, n_jobs=-1)

search_best_pera.fit(X_train, y_train)
random_forest_model = search_best_pera.best_estimator_

y_pred = random_forest_model.predict(X_test)

In [394]:
# 1. Individual component accuracy
for i, col in enumerate(Y.columns):
    acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"{col:12} : {acc:.2%}")

# 2. Overall component accuracy
component_accuracy = np.mean(y_test.values == y_pred)

print(f"Component Accuracy : {component_accuracy:.2%}")

cpu          : 80.74%
gpu          : 86.30%
mobo         : 81.85%
cpu_cooler   : 87.41%
psu          : 90.37%
storage      : 80.37%
ram          : 84.07%
cabinet      : 81.48%
monitor      : 81.85%
Component Accuracy : 83.83%


## **Cat Model**

In [395]:
# Base CatBoost
cat = CatBoostClassifier(
    verbose=0,
    random_state=42,
    allow_writing_files=False)

# Multi-output
multi_cat = MultiOutputClassifier(cat)


# Hyperparameter grid
param_grid = {
    "estimator__iterations": [100, 200],
    "estimator__depth": [4, 6, 8],
    "estimator__learning_rate": [0.01, 0.1],
    "estimator__l2_leaf_reg": [1, 3, 5]}

# Custom accuracy
def multioutput_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_pred = np.squeeze(y_pred)
    return np.mean(y_true == y_pred)

custom_accuracy = make_scorer(multioutput_accuracy)

# GridSearch
cat_model = GridSearchCV(
    estimator=multi_cat,
    param_grid=param_grid,
    cv=5,
    scoring=custom_accuracy,
    n_jobs=-1,
    verbose=0,
    error_score="raise")

# Train
cat_model.fit(X_train, y_train)

KeyboardInterrupt: 

In [ ]:
print(f"Best CV Score: {cat_model.best_score_:.2%}")
cat_model = cat_model.best_estimator_

In [ ]:
y_pred = np.column_stack([
    np.asarray(est.predict(X_test)).ravel()
    for est in cat_model.estimators_])

print(y_pred.shape)  # (288, 9)

# 1. Individual component accuracy — this works fine, each column is single-output
for i, col in enumerate(Y.columns):
    acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"{col:12} : {acc:.2%}")

# 2. Overall component accuracy (element-wise match rate across all targets)
component_accuracy = np.mean(y_test.values == y_pred)
print(f"Component Accuracy : {component_accuracy:.2%}") 

## **Final Training**

In [ ]:
# 1. MAIN DATASET LOAD + CLEAN
df = pd.read_excel("dataset/rigbuilder_main_dataset.xlsx")
df.columns = df.columns.str.strip()
df.drop(columns=['used_amount_saved', 'build_type'], axis=1, inplace=True)
df = df[df['gpu'] != 0].reset_index(drop=True)

df['used_parts'] = df['used_parts'].replace({
    'yes ': 'yes', 'New': 'no', 'No': 'no', 'Yes': 'yes',
    'Under Warranty Used CPU GPU Storage ': 'yes',
    'Under Warranty Used CPU Storage ': 'yes',
    'Under Warranty Used CPU GPU Storage': 'yes',
    'Under Warranty Used CPU Storage': 'yes'})
 
df['monitor_required'] = df['monitor_required'].replace({'No': 'no', 'Yes': 'yes'})
df['upgrade_path'] = df['upgrade_path'].replace({'Medium': 'medium', 'Low': 'Small', 'good': 'medium'})
 
df['monitor_resolution'] = df['monitor_resolution'].replace({
    'FHD ': 'FHD', 'HD': 'FHD',           # agar HD ko FHD hi maanna hai -- warna ye line hata do
    'UHD': '4K', '4K (UHD)': '4K',
    '1440p (QHD)': 'QHD', 'QHD Ultrawide': 'UW-QHD',
})
 
df.drop(1, inplace=True)
df.reset_index(drop=True, inplace=True)
df['upgrade_path'].replace({'small ': 'small'}, inplace=True)
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
 
df["used_parts"] = df["used_parts"].map({"yes": 1, "no": 0})
df["monitor_required"] = df["monitor_required"].map({"yes": 1, "no": 0})
 
df["monitor_size"] = (
    df["monitor_size"].replace(["none", "no"], np.nan).astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float).fillna(0).astype(int))

In [ ]:
# 2. COMPONENT DATASETS LOAD (SIRF EK BAAR, DOBARA RELOAD NAHI KARNA)
cabinet_df = pd.read_excel("dataset/rigbuilder_cabinet.xlsx")
clr_df = pd.read_excel("dataset/rigbuilder_cpu_cooler.xlsx")
cpu_df = pd.read_excel("dataset/rigbuilder_cpu.xlsx")
mobo_df = pd.read_excel("dataset/rigbuilder_mobo.xlsx")
monitor_df = pd.read_excel("dataset/rigbuilder_monitor.xlsx")
psu_df = pd.read_excel("dataset/rigbuilder_psu.xlsx")
ram_df = pd.read_excel("dataset/rigbuilder_ram.xlsx")
storage_df = pd.read_excel("dataset/rigbuilder_storage.xlsx")
gpu_df = pd.read_excel("dataset/rigbuilder_gpu.xlsx")
 
for d in [cpu_df, gpu_df, mobo_df, clr_df, psu_df, ram_df, storage_df, cabinet_df, monitor_df]:
    d.columns = d.columns.str.strip()


# RAM type format fix
mobo_df['ram_type'] = mobo_df['ram_type'].replace({'DDR-3': 'DDR3', 'DDR-4': 'DDR4', 'DDR-5': 'DDR5'})

In [ ]:
# Socket / chipset normalize (SIRF EK BAAR, is cell ke baad cpu_df ko dobara reload NAHI karna)
mobo_df['socket_no'] = mobo_df['socket_no'].astype(str).str.strip().str.upper().str.replace(' ', '', regex=False)
cpu_df['socket_no'] = cpu_df['socket_no'].astype(str).str.strip().str.upper().str.replace(' ', '', regex=False)
ram_df['mobo_socket_compatibility'] = ram_df['mobo_socket_compatibility'].astype(str).str.strip().str.upper().str.replace(' ', '', regex=False)
clr_df['Compatible Socket'] = clr_df['Compatible Socket'].astype(str).str.strip().str.upper().str.replace(' ', '', regex=False)
mobo_df['chipset'] = mobo_df['chipset'].astype(str).str.strip().str.upper()
 
 
# 3. PURANE SOCKET WALE CPUs HATAO (cpu_df ab NORMALIZED hai, isse use karo — reload NAHI karna)
old_sockets = ['LGA1155', 'LGA1150', 'LGA1151']
 
old_cpu_ids = cpu_df[cpu_df['socket_no'].isin(old_sockets)]['cpu_id'].tolist()

 
cpu_df = cpu_df[~cpu_df['socket_no'].isin(old_sockets)]
cpu_df.reset_index(drop=True, inplace=True)
 
df = df[~df['cpu'].isin(old_cpu_ids)]
df.reset_index(drop=True, inplace=True)
 
 
# 4. DROP ROWS WITH MISSING COMPONENT IDs
cols = ['cpu', 'gpu', 'mobo', 'cpu_cooler', 'psu', 'storage', 'ram', 'cabinet', 'monitor']
# df = df.dropna(subset=cols)
df.reset_index(drop=True, inplace=True)


df[categorical_cols] = df[categorical_cols].astype(str)



df['monitor_resolution'] = df['monitor_resolution'].replace({'nan': "none"})
df['monitor'] = df['monitor'].fillna('none')
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)
df['cpu'] = df['cpu'].apply(lambda x: x.strip())
 
# 5. X, Y BANAO
X = df[['budget', 'used_parts', 'usage_scenario', 'colour_theme',
        'monitor_required', 'monitor_size', 'monitor_resolution',
        'upgrade_path']]
 
Y = df[cols].copy()
 
categorical_cols = ["usage_scenario", "colour_theme", "monitor_resolution", "upgrade_path"]


In [ ]:
encoded_data = ohe.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded_data, columns=ohe.get_feature_names_out(categorical_cols))


X = X.drop(columns=categorical_cols)
X = pd.concat([X, encoded_df], axis=1)
 
# IMPORTANT: saare Y columns string banao (mixed-type crash se bachne ke liye, jaise gpu mein hua tha)
for col in Y.columns:
    Y[col] = Y[col].astype(str)


target_encoders = {}
for col in Y.columns:
    label_encoder = LabelEncoder()
    Y[col] = label_encoder.fit_transform(Y[col])
    target_encoders[col] = label_encoder
 
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
df.shape

(679, 17)

In [ ]:
# 6. RANDOM FOREST TRAIN (natively multi-output support karta hai, MultiOutputClassifier ki zarurat nahi)
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]}
 
rf = RandomForestClassifier(random_state=42)
 
search_best_pera = GridSearchCV(estimator=rf, param_grid=param_dist, cv=5, n_jobs=-1)
search_best_pera.fit(X_train, y_train)
 
random_forest_model = search_best_pera.best_estimator_

In [82]:
# Base CatBoost
cat = CatBoostClassifier(
    verbose=0,
    random_state=42,
    allow_writing_files=False)

multi_cat = MultiOutputClassifier(cat)

param_grid = {
    "estimator__iterations": [100, 200],
    "estimator__depth": [4, 6, 8],
    "estimator__learning_rate": [0.01, 0.1],
    "estimator__l2_leaf_reg": [1, 3, 5]}

# Custom accuracy
def multioutput_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_pred = np.squeeze(y_pred)
    return np.mean(y_true == y_pred)

custom_accuracy = make_scorer(multioutput_accuracy)

# GridSearch
cat_model = GridSearchCV(
    estimator=multi_cat,
    param_grid=param_grid,
    cv=5,
    scoring=custom_accuracy,
    n_jobs=-1,
    verbose=0,
    error_score="raise")

# Train
cat_model.fit(X, Y)
print(f"Best CV Score: {cat_model.best_score_:.2%}")
cat_model = cat_model.best_estimator_

Best CV Score: 22.11%


In [83]:
"models Train 👍"

'models Train 👍'

## **Decode predictions**
prediction → decoded component IDs → actual CPU/GPU/Motherboard names ka flow banayenge.

In [84]:
# Component ID columns
components_id_col_name = {
    "CPU": "cpu_id",
    "GPU": "gpu id",
    "Mother Board": "model_id",
    "Cooler": "Cooler ID",
    "PSU": "PSU ID",
    "Storage": "storage_id",
    "RAM": "ram_id",
    "Cabinet": "Cabinet ID",
    "Monitor": "Monitor ID"}

# Target columns
target_columns = [
    "cpu",
    "gpu",
    "mobo",
    "cpu_cooler",
    "psu",
    "storage",
    "ram",
    "cabinet",
    "monitor"]

# Price columns
price_columns = {
    "GPU": "gpu price",
    "Mother Board": "price",
    "Cooler": "Price",
    "PSU": "Approx Price (INR)",
    "Storage": "price",
    "RAM": "approx_price",
    "Cabinet": "Price",
    "Monitor": "Price",}

# Component DataFrames
component_dataframes = {
    "CPU": cpu_df,
    "GPU": gpu_df,
    "Mother Board": mobo_df,
    "Cooler": clr_df,
    "PSU": psu_df,
    "Storage": storage_df,
    "RAM": ram_df,
    "Cabinet": cabinet_df,
    "Monitor": monitor_df}

### **1. User input preprocessing**

In [85]:
def preprocess_user_input(budget,
                        used_parts,
                        usage_scenario,
                        colour_theme,
                        monitor_required,
                        monitor_size,
                        monitor_resolution,
                        # build_type,
                        upgrade_path):

    user_input = pd.DataFrame([{
        "budget": budget,
        "used_parts": used_parts,
        "usage_scenario": usage_scenario,
        "colour_theme": colour_theme,
        "monitor_required": monitor_required,
        "monitor_size": monitor_size,
        "monitor_resolution": monitor_resolution,
        # "build_type": build_type,
        "upgrade_path": upgrade_path}])

    encoded_data = ohe.transform(
        user_input[categorical_cols])

    encoded_df = pd.DataFrame(
        encoded_data,
        columns=ohe.get_feature_names_out(categorical_cols))

    user_input = user_input.drop(columns=categorical_cols)

    user_input = pd.concat([user_input, encoded_df], axis=1)

    return user_input

### **compatibility Text**

In [86]:
def get_compatible_chipsets(compatible_socket):
    if pd.isna(compatible_socket):
        return []

    return re.findall(r'\b[A-Z]\d{3}\b', str(compatible_socket))

def normalize_str(value):
    return str(value).strip().upper().replace("-", "").replace(" ", "")


def check_cpu_mobo(cpu, mobo):
    cpu_socket = normalize_str(cpu["socket_no"])
    mobo_socket = normalize_str(mobo["socket_no"])

    # Sirf socket match karo — compatible_socket column corrupt hai (Excel references leak hue hain),
    # isliye chipset-level check hata diya, socket match hi reliable compatibility factor hai
    return cpu_socket == mobo_socket


def check_ram_mobo(ram, mobo):

    ram_generation = str(ram["ram_generation"]).strip().upper()

    mobo_ram_type = str(mobo["ram_type"]).strip().upper()

    ram_compatible_sockets = [
        socket.strip().upper()
        for socket in str(ram["mobo_socket_compatibility"]).split("/")]

    mobo_socket = str(mobo["socket_no"]).strip().upper()

    if ram_generation != mobo_ram_type:
        return False

    if mobo_socket not in ram_compatible_sockets:
        return False

    return True


def check_gpu_psu(gpu, psu):

    gpu_required = int(str(gpu["psu required"]).replace("W", "").strip())
    psu_wattage = float(psu["Wattage (W)"])

    return psu_wattage >= gpu_required


def check_gpu_cabinet(gpu, cabinet):

    gpu_length = float(str(gpu["gpu length"]).replace("mm", "").strip())
    max_gpu_length = float(str(cabinet["Max GPU Length"]).replace("mm", "").strip())

    return gpu_length <= max_gpu_length


def get_watt(value):
    return float(str(value).replace("W", "").strip())


def check_cpu_cooler(cpu, cooler):

    cpu_socket = str(cpu["socket_no"]).strip().upper()

    compatible_sockets = [
        socket.strip().upper()
        for socket in str(
            cooler["Compatible Socket"]).split(",")]

    cpu_tdp = get_watt(cpu["tdp_wattage"])

    cooler_tdp = get_watt(cooler["Max Rated TDP"])

    socket_ok = (cpu_socket in compatible_sockets)

    tdp_ok = (cpu_tdp <= cooler_tdp)

    return socket_ok and tdp_ok


# def get_mm(value):
    # return float(str(value).replace("mm", "").strip())
def get_mm(value):
    match = re.search(r'\d+(\.\d+)?', str(value))
    if match:
        return float(match.group())
    return None


def check_cooler_cabinet(cooler, cabinet):

    cooler_height = get_mm(cooler["Cooler Height"])
    max_cooler_height = get_mm(cabinet["Max Cooler Height"])
    return cooler_height <= max_cooler_height


### **check compatibility**

In [87]:
#new
def check_compatibility(build):

    results = {
        "CPU ↔ Motherboard": check_cpu_mobo(
            build["CPU"],
            build["Mother Board"]),

        "RAM ↔ Motherboard": check_ram_mobo(
            build["RAM"],
            build["Mother Board"]),

        "GPU ↔ PSU": check_gpu_psu(
            build["GPU"],
            build["PSU"]),

        "GPU ↔ Cabinet": check_gpu_cabinet(
            build["GPU"],
            build["Cabinet"]),

        "CPU ↔ Cooler": check_cpu_cooler(
            build["CPU"],
            build["Cooler"]),

        "Cooler ↔ Cabinet": check_cooler_cabinet(
            build["Cooler"],
            build["Cabinet"])}

    return results

#new
def is_build_compatible(compatibility_results):
    return all(compatibility_results.values())

### **Find Compatible Component**

In [88]:
# new
def find_compatible_mobos(cpu, mobo_df):
    compatible = []
    for _, mobo in mobo_df.iterrows():
        if check_cpu_mobo(cpu, mobo):
            compatible.append(mobo)
    return compatible


def find_compatible_rams(mobo, ram_df):
    compatible = []
    for _, ram in ram_df.iterrows():
        if check_ram_mobo(ram, mobo):
            compatible.append(ram)
    return compatible


def find_compatible_coolers(cpu, cooler_df):
    compatible = []
    for _, cooler in cooler_df.iterrows():
        if check_cpu_cooler(cpu, cooler):
            compatible.append(cooler)
    return compatible


def find_compatible_psus(gpu, psu_df):
    compatible = []
    for _, psu in psu_df.iterrows():
        if check_gpu_psu(gpu, psu):
            compatible.append(psu)
    return compatible


def find_compatible_cabinets(gpu, cooler, cabinet_df):
    compatible = []
    for _, cabinet in cabinet_df.iterrows():
        if check_gpu_cabinet(gpu, cabinet) and check_cooler_cabinet(cooler, cabinet):
            compatible.append(cabinet)
    return compatible


def pick_closest_by_price(candidates, target_price, price_col):
    if not candidates:
        return None

    def price_diff(item):
        price = pd.to_numeric(item[price_col], errors="coerce")
        if pd.isna(price):
            return float("inf")
        return abs(price - target_price)

    return min(candidates, key=price_diff)

### **Resolve Compatible Build**

In [89]:
def resolve_compatible_build(cpu, gpu, budget, component_dataframes, price_columns):
    build = {"CPU": cpu, "GPU": gpu}

    mobo_candidates = find_compatible_mobos(cpu, component_dataframes["Mother Board"])
    if not mobo_candidates:
        raise ValueError(f"No compatible motherboard found for CPU {cpu.get('cpu_id')}")

    # Motherboards ko price-closeness ke hisaab se sort karo
    target_mobo_price = budget * 0.15
    mobo_candidates.sort(
        key=lambda m: abs(pd.to_numeric(m[price_columns["Mother Board"]], errors="coerce") - target_mobo_price)
        if pd.notna(pd.to_numeric(m[price_columns["Mother Board"]], errors="coerce")) else float("inf"))

    # Har motherboard try karo jab tak RAM na mile
    selected_mobo = None
    selected_ram = None
    for mobo in mobo_candidates:
        ram_candidates = find_compatible_rams(mobo, component_dataframes["RAM"])
        if ram_candidates:
            selected_mobo = mobo
            selected_ram = pick_closest_by_price(ram_candidates, budget * 0.1, price_columns["RAM"])
            break

    if selected_mobo is None:
        raise ValueError("Koi bhi compatible motherboard RAM ke saath nahi mila")

    build["Mother Board"] = selected_mobo
    build["RAM"] = selected_ram

    # Cooler
    cooler_candidates = find_compatible_coolers(cpu, component_dataframes["Cooler"])
    if not cooler_candidates:
        raise ValueError("No compatible cooler found for CPU")
    build["Cooler"] = pick_closest_by_price(cooler_candidates, budget * 0.05, price_columns["Cooler"])

    # PSU
    psu_candidates = find_compatible_psus(gpu, component_dataframes["PSU"])
    if not psu_candidates:
        raise ValueError("No compatible PSU found for GPU")
    build["PSU"] = pick_closest_by_price(psu_candidates, budget * 0.1, price_columns["PSU"])

    # Cabinet
    cabinet_candidates = find_compatible_cabinets(gpu, build["Cooler"], component_dataframes["Cabinet"])
    if not cabinet_candidates:
        raise ValueError("No compatible cabinet found for GPU + Cooler combo")
    build["Cabinet"] = pick_closest_by_price(cabinet_candidates, budget * 0.05, price_columns["Cabinet"])

    return build

### **Get Compatibility Summary**

In [90]:
def get_compatibility_summary(compatibility_results):

    issues = [
        component
        for component, result
        in compatibility_results.items()
        if not result]

    return {"is_compatible": len(issues) == 0,
            "issues": issues}

### **2. Model prediction**

In [91]:
def predict_components(user_input, model):

    prediction = model.predict(user_input)
    prediction = np.squeeze(prediction)
    return prediction

### **3. Encoded predictions ko actual IDs mein decode karna**

In [92]:
def decode_predictions(prediction):

    decoded_prediction = {}

    for col, encoded_value in zip(target_columns, prediction):

        decoded_value = target_encoders[col].inverse_transform([encoded_value])[0]
        decoded_prediction[col] = decoded_value
        
    return decoded_prediction

### **4. Actual component lookup**

In [93]:
def get_component(df, column_name, component_id):

    result = df[df[column_name] == component_id]

    if result.empty:
        raise ValueError(f"Component {component_id} not found in column {column_name}")

    return result.iloc[0]

In [94]:
def get_component_details(decoded_prediction):

    build = {}

    for component_name, df in component_dataframes.items():

        target_column = {
            "CPU": "cpu",
            "GPU": "gpu",
            "Mother Board": "mobo",
            "Cooler": "cpu_cooler",
            "PSU": "psu",
            "Storage": "storage",
            "RAM": "ram",
            "Cabinet": "cabinet",
            "Monitor": "monitor"}[component_name]

        component_id = decoded_prediction[target_column]

        id_column = components_id_col_name[
            component_name]

        build[component_name] = get_component(
            df,
            id_column,
            component_id)

    return build

### **5. Price calculation**

In [95]:
def calculate_build_price(build):

    price_columns = {
        "CPU": "price",
        "GPU": "gpu price",
        "Mother Board": "price",
        "Cooler": "Price",
        "PSU": "Approx Price (INR)",
        "Storage": "price",
        "RAM": "approx_price",
        "Cabinet": "Price",
        "Monitor": "Approx Price"}

    total = 0

    for component, item in build.items():

        price_column = price_columns[component]

        price = pd.to_numeric(
            item[price_column],
            errors="coerce")

        if pd.notna(price):
            total += price

    return total

### **6. Final recommendation function**

In [96]:
def recommend_pc(
    budget,
    used_parts,
    usage_scenario,
    colour_theme,
    monitor_required,
    monitor_size,
    monitor_resolution,
    upgrade_path,
    model):

    # 1. Prepare user input
    user_input = preprocess_user_input(
        budget,
        used_parts,
        usage_scenario,
        colour_theme,
        monitor_required,
        monitor_size,
        monitor_resolution,
        upgrade_path)


    # 2. Predict component classes
    prediction = predict_components(user_input, model)


    # 3. Decode classes → component IDs
    decoded_prediction = decode_predictions(prediction)


    # 4. Component IDs → actual component rows
    # build = get_component_details(decoded_prediction)
    # new
        # 4. CPU & GPU seedha ML prediction se lo (anchors)
    cpu = get_component(cpu_df, "cpu_id", decoded_prediction["cpu"])
    gpu = get_component(gpu_df, "gpu id", decoded_prediction["gpu"])

    # 4a. Baaki components resolver se guaranteed-compatible milenge
    build = resolve_compatible_build(cpu, gpu, budget, component_dataframes, price_columns)

    # 4b. Storage/Monitor pe hard compatibility rule nahi hai, ML se hi lo
    build["Storage"] = get_component(storage_df, "storage_id", decoded_prediction["storage"])
    build["Monitor"] = get_component(monitor_df, "Monitor ID", decoded_prediction["monitor"])


    # 6. Compatibility validation
    compatibility = check_compatibility(build)
    build_compatible = is_build_compatible(compatibility) #new

    summary = get_compatibility_summary(compatibility) #new

    # 5. Calculate total price
    total_price = calculate_build_price(build)

    return {
        "components": build,
        "total_price": total_price,
        "compatibility": compatibility,
        "build_compatible": build_compatible, #new
        "issues": summary["issues"] #new
}

### **Input Test**

In [ ]:
# cat_model = joblib.load('models/cat_model.pkl')
print(df['usage_scenario'].unique())
print(df['colour_theme'].unique())
print(df['monitor_required'].unique())
print(df['monitor_size'].unique())
print(df['monitor_resolution'].unique())
print(df['upgrade_path'].unique())


['High-End Gaming + Mid Workstation' 'Budget Gaming + Office Usage'
 'Entry Gaming + Office Usage' 'Mid-Tier Gaming + Office Work'
 'Ultra Gaming + Workstation' 'High-End Gaming + Workstation'
 'Mid-Tier Gaming + Workstation']
['white' 'black' 'stealth' 'none' 'White' 'Black' 'Black/Red'
 'Stealth Black' 'Black/Blue' 'Black/White' 'White/Black' 'White/Blue'
 'Black/Silver' 'All White' 'Black/Grey' 'Black Stealth'
 'Dual Chamber Glass' 'Dark Grey' 'All Black' 'Black/RGB' 'White Stealth'
 'Dual-Tone' 'RGB' 'Stealth']
[1. 0.]
[24 27  0 19 20 22 32 28 34 38 49 31]
['FHD' 'none' 'QHD' '4K' 'UW-QHD']
['medium' 'Medium-High' 'High' 'Small']


In [ ]:
df['']

In [108]:
result = recommend_pc(
    budget=50000,
    used_parts=1,
    usage_scenario="Ultra Gaming + Workstation",
    colour_theme="black",
    monitor_required=0,
    monitor_size=0 ,
    monitor_resolution="none",
    upgrade_path="Medium-High",
    model=cat_model)

ValueError: Component none not found in column Monitor ID

In [105]:
print(result['compatibility'])
print(result['issues'])
print(result['total_price'])

{'CPU ↔ Motherboard': True, 'RAM ↔ Motherboard': True, 'GPU ↔ PSU': True, 'GPU ↔ Cabinet': True, 'CPU ↔ Cooler': True, 'Cooler ↔ Cabinet': True}
[]
41838


In [43]:
df['monitor_required'].unique()

array([1., 0.])

## **Saving Environment**

In [41]:
joblib.dump(ohe, "models/ohe.pkl")

# Target encoders
joblib.dump(target_encoders, "models/target_encoders.pkl")

# Feature columns
joblib.dump(X.columns.tolist(), "models/feature_columns.pkl")

joblib.dump(cat_model, "models/cat_model.pkl")
joblib.dump(random_forest_model, "models/random_forest_model.pkl")

['models/random_forest_model.pkl']

In [ ]:

# # database se data retrieve karna verify 
# @app.route("/test-build")
# def test_build():
#     cpus = CPU.query.limit(5).all()
#     result = []
#     for cpu in cpus:
#         result.append({
#             "id": cpu.cpu_id,
#             "model": cpu.cpu_model,
#             "price": cpu.price,
#             "tier": cpu.cpu_tier})
#     return result



# @app.route("/build")
# def build():
#     budget = 5000
#     cpus = CPU.query.filter(
#     CPU.price <= budget
#     ).order_by(
#     CPU.cpu_tier.desc()
#     ).all()
#     result = []
#     for cpu in cpus:
#         result.append({
#             "id": cpu.cpu_id,
#             "model": cpu.cpu_model,
#             "price": cpu.price,
#             "tier": cpu.cpu_tier})
#     return result

